### Structured Output
Models can be requested to provide their response in a format matching a given schema. This is usefull for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic
pydantic models provides the richest feature set field validation, description and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D8EBE36AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D8EBE371C0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field


class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(descriptio="the movie rating out of 10")

C:\Users\himan\AppData\Local\Temp\ipykernel_9680\857011634.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptio'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  rating:float=Field(descriptio="the movie rating out of 10")


In [3]:
model_with_strucutred=model.with_structured_output(Movie)
model_with_strucutred

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018FAE84A110>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000018FAE8F1870>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'descriptio': 'the movie rating out o

In [4]:
model.invoke("provide details about the inception")

AIMessage(content='<think>\nOkay, I need to provide details about the inception of something. But wait, the user just said "provide details about the inception". The word "inception" can have different meanings depending on the context. Let me think... The most obvious one is the movie "Inception" directed by Christopher Nolan. But maybe they\'re referring to the concept of inception as an idea or process of starting something. Hmm.\n\nFirst, I should check if there\'s any context given. The user hasn\'t provided any additional information. So, to cover all bases, I should explain both possibilities: the movie and the general concept. That way, the user gets comprehensive information regardless of their intent.\n\nStarting with the movie. Let me recall the details. "Inception" was released in 2010, directed by Christopher Nolan. It\'s a sci-fi action film starring Leonardo DiCaprio as Dom Cobb, a professional thief who steals secrets by infiltrating the subconscious. The plot revolves 

In [10]:
model_with_strucutred.invoke("provide details about the inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output alogside parsed structure

In [2]:
from pydantic import BaseModel,Field


class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(descriptio="the movie rating out of 10")

model_with_strucutred= model.with_structured_output(Movie, include_raw=True)

response=model_with_strucutred.invoke("provide details about the inception")
response

C:\Users\himan\AppData\Local\Temp\ipykernel_25516\2365790868.py:9: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descriptio'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  rating:float=Field(descriptio="the movie rating out of 10")


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception." Let me check the tools provided. There\'s a function called Movie that requires title, year, director, and rating. I need to gather that information for Inception. The title is obviously "Inception." The director is Christopher Nolan. It was released in 2010, so the year is 2010. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it 8.8/10. So I should structure the tool call with those parameters. Make sure all required fields are included and the types are correct: title and director as strings, year as integer, rating as a number. No typos. Alright, that should do it.\n', 'tool_calls': [{'id': 'zceyq28n0', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 216

### Nested Structure

In [3]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    tile:str
    year: int
    cast: list[Actor]
    genres: list[str]
    bugget: float | None =Field(None, description="Bugget details in millions USD")

model_with_strucutred=model.with_structured_output(MovieDetails)

response=model_with_strucutred.invoke("Provide details about the movie Inception")
response

MovieDetails(tile='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Science Fiction', 'Action', 'Thriller'], bugget=160.0)